# BraTS 2021 3D UNet Baseline — 65 Epochs Pure FP32 Runner on Kaggle

Notebook chuẩn hóa chạy **Standard 3D UNet Baseline trên 350 ca BraTS 2021 (280 Train / 35 Val / 35 Held-Out Test)**.

### Đặc tả cấu hình chạy:
- **Độ chính xác**: Thuần **`FP32` (`--no-amp`)** loại bỏ 100% nguy cơ tràn số `NaN` ở `BatchNorm3d` trên GPU T4.
- **Phần cứng**: **`GPU T4 x2`** (`torch.nn.DataParallel` tự động kích hoạt) + 4 CPU workers (`persistent_workers=True`).
- **Lịch trình huấn luyện**: **`65` epochs**, 5 mốc Validation (`val_interval=13`: epochs 13, 26, 39, 52, 65).
- **Dữ liệu**: Nhấn **`+ Add Input`** -> tìm **`brats-2021-task1`** (`dschettler8845/brats-2021-task1`). Script tự động giải nén vào `/kaggle/tmp` (SSD tạm), tiêu tốn **0 MB** quota `/kaggle/working`.
- **Thời gian chạy**: ~2.1 giờ trên 2x GPU T4 (hoàn toàn an toàn trong hạn mức 12h của Kaggle).

In [ ]:
# 1. Kiểm tra GPU, RAM, ổ đĩa /kaggle/tmp và Input Dataset
!nvidia-smi
!df -h /kaggle/working /tmp /dev/shm
!find /kaggle/input -maxdepth 3

In [ ]:
# 2. Cài đặt MONAI + Clone repository vào /kaggle/tmp/repo (giữ /kaggle/working sạch gọn chỉ chứa kết quả!)
!pip install -q monai nibabel scipy matplotlib tqdm
!rm -rf /kaggle/tmp/repo && mkdir -p /kaggle/tmp
# Clone từ branch baselineTrain của nhóm (hoặc fallback về repo cá nhân)
!git clone -b baselineTrain https://github.com/nam21-12-06/test.git /kaggle/tmp/repo || git clone https://github.com/chitruonghf711/brats2021-3d-unet-study.git /kaggle/tmp/repo

In [ ]:
# 3. Huấn luyện Standard 3D UNet Baseline (350 ca BraTS 2021: 280 Train / 35 Val / 35 Held-Out Test)
#    - Chạy thuần FP32 (--no-amp) chống tràn số NaN
#    - DataParallel 2x T4 + 4 CPU workers + CosineAnnealingLR 65 epochs
#    - Sau khi hoàn thành 65 epochs, tự động nạp best_metric_model.pth để đánh giá Held-Out Test (35 ca)
!python -u /kaggle/tmp/repo/scripts/run_unet_baseline.py \
    --max-samples 350 \
    --epochs 65 \
    --val-interval 13 \
    --roi-size 96 \
    --batch-size 4 \
    --sw-batch-size 4 \
    --lr 1e-3 \
    --loss-type dice_bce \
    --overlap 0.5 \
    --cache-rate 0.0 \
    --num-workers 4 \
    --no-amp \
    --keep-latest \
    --output-dir /kaggle/working/outputs/unet_baseline \
    --checkpoint-dir /kaggle/working/checkpoints/unet_baseline

In [ ]:
# 5. Hiển thị trực tiếp kết quả Held-Out Test & Biểu đồ ngay trong Kaggle Output
import json
from IPython.display import Image, display

report_path = "/kaggle/working/outputs/unet_baseline/baseline_metrics_report.json"
with open(report_path, "r", encoding="utf-8") as f:
    rep = json.load(f)

print(f"Best Validation Epoch: {rep['best_val_epoch']} | Best Val Mean Dice: {rep['best_val_dice_mean']:.4f}")
print("Held-Out Test Summary (Mean):", json.dumps(rep["test_summary_mean"], indent=2))
print("Held-Out Test Distribution (Mean/Std/Median):", json.dumps(rep["test_distribution_stats"], indent=2))

display(Image(filename="/kaggle/working/outputs/unet_baseline/baseline_learning_curves.png"))
display(Image(filename="/kaggle/working/outputs/unet_baseline/baseline_test_predictions.png"))